In [1]:
from pathlib import Path
import json
import os
import time
from dotenv import load_dotenv
from datetime import datetime, timezone


def find_project_root():
    current = Path.cwd().resolve()

    for path in [current, *current.parents]:
        if (path / "prompts" / "system_v0.md").exists():
            return path

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto TI-support-RAG."
    )


PROJECT_ROOT = find_project_root()

PROMPT_PATH = PROJECT_ROOT / "prompts" / "system_v0.md"
TEST_PATH = PROJECT_ROOT / "test" / "test_cases.json"
RESULTS_PATH = PROJECT_ROOT / "docs" / "results.json"

MODEL = "openai/gpt-oss-20b"
PROMPT_VERSION = "system_v0"

print(f"Proyecto:   {PROJECT_ROOT}")
print(f"Prompt:     {PROMPT_PATH}")
print(f"Casos:      {TEST_PATH}")
print(f"Resultados: {RESULTS_PATH}")
print(f"Modelo:     {MODEL}")
print(f"Versión:    {PROMPT_VERSION}")


Proyecto:   /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG
Prompt:     /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG/prompts/system_v0.md
Casos:      /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG/test/test_cases.json
Resultados: /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG/docs/results.json
Modelo:     openai/gpt-oss-20b
Versión:    system_v0


In [2]:
load_dotenv(PROJECT_ROOT / ".env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError(
        "No se encontró GROQ_API_KEY en el archivo .env."
    )

print("GROQ_API_KEY configurada: True")


GROQ_API_KEY configurada: True


In [3]:
with open(PROMPT_PATH, "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

with open(TEST_PATH, "r", encoding="utf-8") as f:
    TEST_CASES = json.load(f)

print(f"Prompt cargado: {len(SYSTEM_PROMPT)} caracteres")
print(f"Casos de prueba cargados: {len(TEST_CASES)}")

for case in TEST_CASES:
    print(f"- {case['id']}: {case['tipo']}")


Prompt cargado: 3687 caracteres
Casos de prueba cargados: 5
- case_01: normal
- case_02: ambiguo
- case_03: incompleto
- case_04: malicioso
- case_05: fuera_de_alcance


In [4]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from schemas.request_v1 import SolicitudTI

print("Schema SolicitudTI cargado correctamente.")
print("Campos del contrato:")

for field_name, field_info in SolicitudTI.model_fields.items():
    print(f"- {field_name}: {field_info.annotation}")


Schema SolicitudTI cargado correctamente.
Campos del contrato:
- categoria: typing.Literal['hardware', 'software', 'redes', 'cuentas', 'seguridad', 'acceso', 'otros']
- prioridad: typing.Literal['baja', 'media', 'alta']
- resumen: <class 'str'>
- datos_faltantes: list[typing.Annotated[str, Strict(strict=True)]]
- requiere_humano: <class 'bool'>
- confianza: <class 'float'>


In [5]:
from pydantic import ValidationError


def validate_output(data):
    try:
        validated = SolicitudTI.model_validate(data)

        return True, [], validated

    except ValidationError as error:
        errors = []

        for item in error.errors():
            location = ".".join(str(part) for part in item["loc"])
            message = item["msg"]

            errors.append(
                f"{location}: {message}"
            )

        return False, errors, None


print("Validación mediante Pydantic definida correctamente.")


Validación mediante Pydantic definida correctamente.


In [6]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

print("Cliente Groq inicializado correctamente.")


Cliente Groq inicializado correctamente.


In [7]:
def call_groq(user_text):
    start_time = time.perf_counter()

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        max_tokens=500,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": (
                    "<texto_usuario>\n"
                    f"{user_text}\n"
                    "</texto_usuario>"
                ),
            },
        ],
        response_format={
            "type": "json_object"
        },
    )

    elapsed = time.perf_counter() - start_time

    raw_output = response.choices[0].message.content
    finish_reason = response.choices[0].finish_reason

    usage = None

    if response.usage:
        usage = {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
        }

    return {
        "raw_output": raw_output,
        "finish_reason": finish_reason,
        "latency_seconds": round(elapsed, 4),
        "usage": usage,
    }


print("Función call_groq() definida correctamente.")


Función call_groq() definida correctamente.


In [8]:
test_response = call_groq(
    "Mi computador no enciende desde esta mañana."
)

print("SALIDA RAW:")
print(test_response["raw_output"])

print("\nRAZÓN DE FINALIZACIÓN:")
print(test_response["finish_reason"])

print("\nLATENCIA:")
print(test_response["latency_seconds"], "segundos")

print("\nUSO DE TOKENS:")
print(test_response["usage"])


SALIDA RAW:
{"category":"hardware","priority":"alta","summary":"Computador no enciende desde esta mañana.","missing_data":["Modelo del computador","Ubicación física","Estado de la fuente de alimentación","Si hay luces indicadoras o sonidos al intentar encender"],"requires_human_intervention":true,"confidence":0.9}

RAZÓN DE FINALIZACIÓN:
stop

LATENCIA:
1.0287 segundos

USO DE TOKENS:
{'prompt_tokens': 882, 'completion_tokens': 233, 'total_tokens': 1115}


In [9]:
parsed_response = json.loads(
    test_response["raw_output"]
)

validation_ok, validation_errors, validated_output = (
    validate_output(parsed_response)
)

print("VALIDACIÓN:", validation_ok)

if validation_errors:
    print("\nERRORES ENCONTRADOS:")

    for error in validation_errors:
        print(f"- {error}")

else:
    print("\nLa salida cumple el contrato.")

    print("\nSALIDA VALIDADA:")

    print(validated_output.model_dump())


VALIDACIÓN: False

ERRORES ENCONTRADOS:
- categoria: Field required
- prioridad: Field required
- resumen: Field required
- datos_faltantes: Field required
- requiere_humano: Field required
- confianza: Field required
- category: Extra inputs are not permitted
- priority: Extra inputs are not permitted
- summary: Extra inputs are not permitted
- missing_data: Extra inputs are not permitted
- requires_human_intervention: Extra inputs are not permitted
- confidence: Extra inputs are not permitted


In [10]:
def run_case(case):
    result = {
        "id": case["id"],
        "tipo": case["tipo"],
        "input": case["input"],
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "raw_output": None,
        "parsed_output": None,
        "validated_output": None,
        "validation_ok": False,
        "validation_errors": [],
        "technical_error": None,
        "decision": None,
        "latency_seconds": None,
        "usage": None,
        "finish_reason": None,
    }

    try:
        response = call_groq(case["input"])

        result["raw_output"] = response["raw_output"]
        result["latency_seconds"] = response["latency_seconds"]
        result["usage"] = response["usage"]
        result["finish_reason"] = response["finish_reason"]

        try:
            parsed_output = json.loads(
                response["raw_output"]
            )

        except json.JSONDecodeError as error:
            result["technical_error"] = (
                f"JSON inválido: {error}"
            )
            result["decision"] = "ERROR_TECNICO"
            return result

        result["parsed_output"] = parsed_output

        (
            validation_ok,
            validation_errors,
            validated_output,
        ) = validate_output(parsed_output)

        result["validation_ok"] = validation_ok
        result["validation_errors"] = validation_errors

        if validated_output is not None:
            result["validated_output"] = (
                validated_output.model_dump()
            )

        if validation_ok:
            result["decision"] = "OK_VALIDADO"
        else:
            result["decision"] = "ERROR_FORMATO"

    except Exception as error:
        result["technical_error"] = str(error)
        result["decision"] = "ERROR_TECNICO"

    return result


results = []

for case in TEST_CASES:
    print(
        f"Ejecutando {case['id']} ({case['tipo']})..."
    )

    result = run_case(case)
    results.append(result)

    print(
        f"  Decisión: {result['decision']} | "
        f"Validación: {result['validation_ok']} | "
        f"Latencia: {result['latency_seconds']} s"
    )

print("\nPrueba completa.")
print(f"Casos ejecutados: {len(results)}")


Ejecutando case_01 (normal)...
  Decisión: ERROR_FORMATO | Validación: False | Latencia: 0.6557 s
Ejecutando case_02 (ambiguo)...
  Decisión: ERROR_FORMATO | Validación: False | Latencia: 0.7133 s
Ejecutando case_03 (incompleto)...
  Decisión: ERROR_FORMATO | Validación: False | Latencia: 0.6076 s
Ejecutando case_04 (malicioso)...
  Decisión: ERROR_TECNICO | Validación: False | Latencia: None s
Ejecutando case_05 (fuera_de_alcance)...
  Decisión: ERROR_FORMATO | Validación: False | Latencia: 0.9208 s

Prueba completa.
Casos ejecutados: 5


In [11]:
from pprint import pprint

for result in results:
    print("\n" + "=" * 70)
    print(f"{result['id']} — {result['tipo']}")
    print(f"Decisión: {result['decision']}")
    print(f"Latencia: {result['latency_seconds']} s")
    print(f"Finish reason: {result['finish_reason']}")

    print("\nSalida parseada:")
    pprint(result["parsed_output"])

    if result["validated_output"] is not None:
        print("\nSalida validada:")
        pprint(result["validated_output"])

    if result["validation_errors"]:
        print("\nErrores de validación:")
        for error in result["validation_errors"]:
            print(f"- {error}")

    if result["technical_error"]:
        print("\nError técnico:")
        print(result["technical_error"])



case_01 — normal
Decisión: ERROR_FORMATO
Latencia: 0.6557 s
Finish reason: stop

Salida parseada:
{'category': 'hardware',
 'confidence': 0.9,
 'missing_data': ['Modelo del computador',
                  'Estado de la fuente de alimentación',
                  'Presencia de luces indicadoras o sonidos de arranque'],
 'priority': 'alta',
 'requires_human_intervention': True,
 'summary': 'Computador no enciende desde esta mañana.'}

Errores de validación:
- categoria: Field required
- prioridad: Field required
- resumen: Field required
- datos_faltantes: Field required
- requiere_humano: Field required
- confianza: Field required
- category: Extra inputs are not permitted
- priority: Extra inputs are not permitted
- summary: Extra inputs are not permitted
- missing_data: Extra inputs are not permitted
- requires_human_intervention: Extra inputs are not permitted
- confidence: Extra inputs are not permitted

case_02 — ambiguo
Decisión: ERROR_FORMATO
Latencia: 0.7133 s
Finish reason: stop

In [15]:
execution = {
    "prompt_version": PROMPT_VERSION,
    "model": MODEL,
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "total_cases": len(results),
    "results": results,
}

history = {
    "executions": [
        execution
    ]
}

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        history,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Resultados guardados en: {RESULTS_PATH}")
print(f"Ejecuciones almacenadas: {len(history['executions'])}")

Resultados guardados en: /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG/docs/results.json
Ejecuciones almacenadas: 1
